In [1]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cross_decomposition import CCA
from scipy.stats import pearsonr 
from scipy import stats

###############################################################################
# Paths & constants
###############################################################################
PUPIL_ROOT = Path(r"C:\Users\cdd\Documents\Uni\Special_course\pupil_processed_clara")
EEG_ROOT   = Path(r"C:\Users\cdd\Documents\Uni\Special_course\code\Special_course\eeg_theta_processed2")

FRONTAL_MIDLINE = ['AFz','AF3','AF4','Fz','F1','F2','F3','F4','FC3','FC1','FC2','FC4','Cz','C3','C1','C2','C4']

OUT_TRIALS  = Path("trial_level_cca_fixedlag.csv")
OUT_SUBJECT = Path("subject_best_lag.csv")

SUBJECTS = np.setdiff1d(np.arange(32, 99), [32, 37, 53, 61, 66, 78, 84, 90, 94, 96])
#SUBJECTS = np.setdiff1d(np.arange(32, 99), [32, 37, 53, 61, 66, 78, 84, 90, 94, 96])
SHIFTS = np.arange(-100, 101)      # ±1 s at 100 Hz → samples
WIN_OFFSET1 = 200                  # discard first 200 ms
WIN_OFFSET2 = 110                  # discard last 110 ms

###############################################################################
# Helper functions
###############################################################################

def normalise_eeg(x: np.ndarray) -> np.ndarray:
    """Centre each channel and scale so Σ x² = 1 over time×channels."""
    x = x - x.mean(axis=0, keepdims=True)
    scale = np.sqrt(np.mean(x**2))
    return x / scale


def cca_corr(eeg: np.ndarray, pupil: np.ndarray) -> float:
    """Canonical correlation (single component)."""
    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(eeg, pupil)
    u, v = cca.transform(eeg, pupil)
    return float(np.corrcoef(u[:, 0], v[:, 0])[0, 1])


In [2]:
def load_all_trials(
        sub: int,
        eeg_root: Path = EEG_ROOT,
        pupil_root: Path = PUPIL_ROOT,
        min_len: int = 40,
        max_len_diff: int = 30,
) -> list[tuple[np.ndarray, np.ndarray, dict]]:
    """
    Load *all* valid EEG-pupil trial pairs for one subject.

    Parameters
    ----------
    sub : int
        Numeric subject ID (e.g. 42).
    eeg_root, pupil_root : Path
        Roots of the pre-processed EEG and pupil folders.
    min_len : int
        Minimum number of samples a pupil trace must have to be accepted.
    max_len_diff : int
        Reject trial if |len(pupil)-len(eeg)| exceeds this.
    Returns
    -------
    trials : list of (eeg, pupil_z, meta)
        * eeg        - (T × n_channels) float64, already centred/scaled
        * pupil_z    - (T × 1) float64, per-trial z-scored
        * meta       - dict with subject/condition/load/epoch
    """
    trials = []
    sub_tag = f"sub-{sub:03d}"
    eeg_sub  = eeg_root   / sub_tag
    pupil_sub = pupil_root / sub_tag

    if not eeg_sub.exists():
        print(f"{sub_tag}: EEG folder missing - skipped")
        return trials

    # iterate condition (“control” / “memory”) and load (“05” / “09” / “13”)
    for cond_path in sorted(eeg_sub.iterdir()):
        if not cond_path.is_dir():
            continue
        for load_path in sorted(cond_path.iterdir()):
            if not load_path.is_dir():
                continue

            # matching pupil directory
            pupil_path = pupil_sub / cond_path.name / load_path.name
            if not pupil_path.exists():
                continue

            eeg_epochs   = sorted(load_path.glob("trial_*.csv"))
            pupil_epochs = sorted(pupil_path.glob("trial_*.csv"))
            common = {f.name for f in eeg_epochs} & {f.name for f in pupil_epochs}
            if not common:
                continue

            for fname in sorted(common):
                eeg_df = pd.read_csv(load_path / fname, comment="#", index_col=0)
                pupil_df = pd.read_csv(pupil_path / fname, comment="#",
                                       names=["time", "diameter_z"], index_col=0)

                eeg   = eeg_df.values.astype(float)
                pupil = pupil_df["diameter_z"].values.astype(float)

                # basic validity checks
                if len(pupil) < min_len or abs(len(pupil) - len(eeg)) > max_len_diff:
                    continue

                # normalise signals ----------------------------------------
                eeg_norm = normalise_eeg(eeg)           # your helper from before
                pupil_z  = ((pupil - pupil.mean()) / pupil.std(ddof=0))

                # same number of samples
                T = min(len(eeg_norm), len(pupil_z))
                eeg_norm = eeg_norm[0:T, :]  # (T × n_channels)
                pupil_z  = pupil_z[0:T].reshape(-1, 1)

                meta = {
                    "subject":   sub_tag,
                    "condition": cond_path.name,
                    "load":      int(load_path.name),
                    "epoch":     fname
                }
                trials.append((eeg_norm, pupil_z, meta))

    return trials

from typing import List, Tuple
import numpy as np

def split_trials_by_condition(
        trials: List[Tuple[np.ndarray, np.ndarray, dict]],
        memory_label: str = "memory",
        control_label: str = "control"
) -> Tuple[List[Tuple[np.ndarray, np.ndarray, dict]], List[Tuple[np.ndarray, np.ndarray, dict]]]:
    """
    Separate a mixed list of (eeg, pupil, meta) trial tuples into memory-condition and control-condition sub-lists.

    Parameters
    ----------
    trials : list of tuples
        Each tuple = (eeg_array, pupil_array, meta_dict).
        meta_dict must contain a key 'condition'.
    memory_label : str
        The value of meta['condition'] that marks a memory trial.
    control_label : str
        The value of meta['condition'] that marks a control trial.

    Returns
    -------
    memory_trials  : list[tuple]
    control_trials : list[tuple]
    """
    memory_trials  = []
    control_trials = []

    for eeg, pupil, meta in trials:
        cond = meta.get("condition", "").lower()
        if cond == memory_label:
            memory_trials.append((eeg, pupil, meta))
        elif cond == control_label:
            control_trials.append((eeg, pupil, meta))
        else: raise ValueError(f"Unknown condition label: {cond}")

    return memory_trials, control_trials


In [3]:
import numpy as np
from typing import List, Tuple, Dict

def search_best_lag(
        train_trials: List[Tuple[np.ndarray, np.ndarray, dict]],
        shifts: np.ndarray = SHIFTS,
        return_curve: bool = False
) -> Tuple[float, int, Dict[int, float] | None]:
    """
    Search for the lag (sample shift) that maximises the mean canonical
    correlation between EEG and pupil traces in a training set.

    Parameters
    ----------
    train_trials : list of (eeg, pupil_z, meta)
        Each eeg  : 2-D array [time × channels or CCA-components]
        Each pupil: 1-D array [time]
    shifts : np.ndarray
        Array of integer lag shifts (positive = EEG is moved forward).
    return_curve : bool, default False
        If True, also return the full {shift: mean_r} dictionary.

    Returns
    -------
    best_corr : float
        Highest mean canonical correlation found.
    best_shift : int
        Shift (samples) that maximised the correlation.
    mean_r_per_shift : dict | None
        Only when `return_curve` is True.
    """
    win = slice(WIN_OFFSET1, -WIN_OFFSET2)         # common window
    mean_r_per_shift: Dict[int, float] = {}

    for s in shifts:                               # <-- loop over *shifts*, not SHIFTS
        rs = []
        for eeg, pupil_z, _ in train_trials:
            eeg_shifted = np.roll(eeg, s, axis=0)[win]
            r = cca_corr(eeg_shifted, pupil_z[win])
            rs.append(r)
        mean_r_per_shift[s] = float(np.mean(rs))   # cast to plain float for JSON-ability

    best_shift = max(mean_r_per_shift, key=mean_r_per_shift.get)
    best_corr  = mean_r_per_shift[best_shift]

    if return_curve:
        return best_corr, best_shift, mean_r_per_shift
    else:
        return best_corr, best_shift, None


In [4]:

from typing import List, Tuple, Optional

# trials  : list of (eeg, pupil_z, meta)   -- the tuples returned by load_all_trials
# shift   : integer sample shift (best_shift)
# win     : slice or None                  -- cropping window (set to None if the
#                                            trials are already pre-trimmed)
def concat_trials(
        trials: List[Tuple[np.ndarray, np.ndarray, dict]],
        shift: int = 0,
        win: Optional[slice] = None
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Concatenate a list of trials into one long matrix pair ready for CCA.

    Returns
    -------
    X  : ndarray, shape (Σ Tᵢ, n_channels)
    Y  : ndarray, shape (Σ Tᵢ, 1)
    """
    X_blocks, Y_blocks = [], []

    for eeg, pupil_z, _ in trials:
        # 1. roll *inside* the trial so samples never cross trial boundaries
        eeg_shift = np.roll(eeg, shift, axis=0)

        # 2. optional windowing (do it once here if you did **not** crop in loader)
        if win is not None:
            eeg_shift = eeg_shift[win]
            pupil_seg = pupil_z[win]
        else:
            pupil_seg = pupil_z          # already trimmed earlier

        # 3. stack
        X_blocks.append(eeg_shift)
        Y_blocks.append(pupil_seg)

    X = np.vstack(X_blocks)
    Y = np.vstack(Y_blocks)
    return X, Y

def iterate_trials(trials, shift, win):
    """Yield (eeg_shifted, pupil_z_windowed, meta) one by one."""
    for eeg, pupil_z, meta in trials:
        eeg_s = np.roll(eeg, shift, axis=0)[win]
        yield eeg_s, pupil_z[win], meta.copy()

import pandas as pd
import json
from pathlib import Path

def save_cca_weights(cca, subject_tag, lag_ms, eeg_ch_names, condition, out_dir=Path("weights")):
    """
    Dump EEG & pupil canonical weights to CSV/JSON for one subject.

    Parameters
    ----------
    cca            : fitted sklearn.cross_decomposition.CCA
    subject_tag    : "sub-042"
    lag_ms         : e.g. -90
    eeg_ch_names   : list[str] same order as columns in your trial matrices
    out_dir        : destination folder (created if missing)
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    # ------- Pupil weight -> JSON ---------------------------------------
    w_pupil_og = float(cca.y_weights_[0, 0])   # scalar in 1-dim pupil case
    w_pupil = 1.0
    with open(out_dir / f"{subject_tag}_pupil_weight_{condition}_{lag_ms:+d}ms.json", "w") as fh:
        json.dump({"weight": w_pupil}, fh, indent=2)
    
    # ------- EEG weights -> tidy CSV ------------------------------------
    w_eeg = cca.x_weights_[:, 0] / w_pupil_og
    w_eeg = pd.Series(w_eeg, index=eeg_ch_names, name="weight")
    w_eeg.index.name = "channel"
    w_eeg.to_csv(out_dir / f"{subject_tag}_eeg_weights_{condition}_{lag_ms:+d}ms.csv")

    print(f"saved weights for {subject_tag} (lag {lag_ms:+d} ms)")



In [5]:
best_shift_by_sub = {33: -5, 34: 32, 35: -8, 36: -59, 38: -1, 39: 2, 40: 100, 41: -10, 42: -80, 43: 70, 44: -10, 45: 42, 46: -68, 47: 100, 48: -63, 49: 46, 50: 91, 51: 100, 52: 81, 54: 63, 55: -38, 56: 23, 57: 36, 58: 46, 59: -7, 60: 100, 62: -9, 63: 0, 64: 53, 65: -6, 67: 2, 68: 96, 69: 28, 70: 16, 71: 74, 72: 0, 73: 38, 74: -100, 75: 32, 76: 100, 77: 76, 79: 80, 80: -1, 81: 64, 82: -44, 83: -52, 85: 37, 86: 100, 87: 24, 88: 100, 89: -100, 91: -5, 92: 41, 93: 100, 95: -7, 97: -56, 98: 69}
print(best_shift_by_sub)

{33: -5, 34: 32, 35: -8, 36: -59, 38: -1, 39: 2, 40: 100, 41: -10, 42: -80, 43: 70, 44: -10, 45: 42, 46: -68, 47: 100, 48: -63, 49: 46, 50: 91, 51: 100, 52: 81, 54: 63, 55: -38, 56: 23, 57: 36, 58: 46, 59: -7, 60: 100, 62: -9, 63: 0, 64: 53, 65: -6, 67: 2, 68: 96, 69: 28, 70: 16, 71: 74, 72: 0, 73: 38, 74: -100, 75: 32, 76: 100, 77: 76, 79: 80, 80: -1, 81: 64, 82: -44, 83: -52, 85: 37, 86: 100, 87: 24, 88: 100, 89: -100, 91: -5, 92: 41, 93: 100, 95: -7, 97: -56, 98: 69}


### Hold Out 50% with independent CCA for memory and control

In [13]:
import random

import matplotlib.pyplot as plt

all_rows = []  # collect all trial-level results here
# best_shift_by_sub = {}  # best shift per subject
DIG_LEN   = 200           # 2 s at 100 Hz
rows_trials = []
rows_subject = []  

for subj in SUBJECTS:
    print(f"Processing subject {subj:02d}...")
    trials = load_all_trials(subj)                         # list of (eeg, pupil)
    trials_memory, trials_control = split_trials_by_condition(trials)

    random.shuffle(trials_memory)
    random.shuffle(trials_control)

    mid_memory = len(trials_memory) // 2
    mid_control = len(trials_control) // 2

    if mid_memory < 1 or mid_control < 1:
        print(f"Subject {subj:02d} skipped - not enough trials.")
        continue

    train_memory = trials_memory[:mid_memory]
    test_memory  = trials_memory[mid_memory:]
    train_control = trials_control[:mid_control]
    test_control  = trials_control[mid_control:]

    best_shift = best_shift_by_sub[subj] if subj in best_shift_by_sub else print("MISTAKE")
    # ----- fit weights ONCE using all train trials at best_shift ---------
    win = slice(WIN_OFFSET1, -WIN_OFFSET2)

    best_shift = int(best_shift)  # convert to int if it was float
    X_mem_train, Y_mem_train = concat_trials(train_memory, shift=best_shift, win=win)
    X_mem_test, Y_mem_test = concat_trials(test_memory, shift=best_shift, win=win)
    X_ctrl_train, Y_ctrl_train = concat_trials(train_control, shift=best_shift, win=win)
    X_ctrl_test, Y_ctrl_test = concat_trials(test_control, shift=best_shift, win=win)
    
    cca_mem = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca_mem.fit(X_mem_train, Y_mem_train)

    cca_ctrl = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca_ctrl.fit(X_ctrl_train, Y_ctrl_train)

    # -------------------------------------------------------------
    # 2) whole‑trial correlation for MEMORY
    # -------------------------------------------------------------
    cvX_m, cvY_m = cca_mem.transform(X_mem_test, Y_mem_test)
    r_mem, p_mem = pearsonr(cvX_m[:, 0], cvY_m[:, 0])

    rows_subject.append({
        "subject":   subj,
        "condition": "memory",
        "lag_ms":    best_shift * 10,
        "r":         float(r_mem),
        "p-value": float(p_mem),
    })

    # -------------------------------------------------------------
    # 3) whole‑trial correlation for CONTROL
    # -------------------------------------------------------------
    cvX_c, cvY_c = cca_ctrl.transform(X_ctrl_test, Y_ctrl_test)
    r_ctrl, p_ctrl = pearsonr(cvX_c[:, 0], cvY_c[:, 0])

    rows_subject.append({
        "subject":   subj,
        "condition": "control",
        "lag_ms":    best_shift * 10,
        "r":         float(r_ctrl),
        "p-value": float(p_ctrl), 
    })

    save_cca_weights(cca_mem, subject_tag=f"sub-{subj:03d}", lag_ms=best_shift*10, condition = "mem", eeg_ch_names=FRONTAL_MIDLINE, out_dir=Path("weights_hold"))
    save_cca_weights(cca_ctrl, subject_tag=f"sub-{subj:03d}", lag_ms=best_shift*10, condition = "ctrl", eeg_ch_names=FRONTAL_MIDLINE, out_dir=Path("weights_hold"))


##############################################################################
# ---- save -----------------------------------------------------------------
##############################################################################

##############################################################################
# 4)  after the loop - save per‑subject correlations
##############################################################################
pd.DataFrame(rows_subject).to_csv("subject_level_cca_fixedlagTHESIShold.csv", index=False)
print("Finished - saved subject-level correlations as subject_level_cca_fixedlagTHESIShold.csv.")


Processing subject 33...
saved weights for sub-033 (lag -50 ms)
saved weights for sub-033 (lag -50 ms)
Processing subject 34...
saved weights for sub-034 (lag +320 ms)
saved weights for sub-034 (lag +320 ms)
Processing subject 35...
saved weights for sub-035 (lag -80 ms)
saved weights for sub-035 (lag -80 ms)
Processing subject 36...
saved weights for sub-036 (lag -590 ms)
saved weights for sub-036 (lag -590 ms)
Processing subject 38...
saved weights for sub-038 (lag -10 ms)
saved weights for sub-038 (lag -10 ms)
Processing subject 39...
saved weights for sub-039 (lag +20 ms)
saved weights for sub-039 (lag +20 ms)
Processing subject 40...
saved weights for sub-040 (lag +1000 ms)
saved weights for sub-040 (lag +1000 ms)
Processing subject 41...
saved weights for sub-041 (lag -100 ms)
saved weights for sub-041 (lag -100 ms)
Processing subject 42...
saved weights for sub-042 (lag -800 ms)
saved weights for sub-042 (lag -800 ms)
Processing subject 43...
saved weights for sub-043 (lag +700 

In [15]:
df_subj = pd.read_csv("subject_level_cca_fixedlagTHESIShold.csv")

# Calculate and print mean correlations for each condition
mean_memory = df_subj[df_subj['condition'] == 'memory']['r'].mean()
mean_control = df_subj[df_subj['condition'] == 'control']['r'].mean()
print(f"Mean correlation for memory condition: {mean_memory:.4f}")
print(f"Mean correlation for control condition: {mean_control:.4f}")
print()

# Pivot so we have columns for memory and control per subject
df_wide = df_subj.pivot(index="subject", columns="condition", values="r")
print(df_wide.head())

# Get degrees of freedom
df = len(df_wide) - 1
print(f"Degrees of freedom: {df}")

# Compute paired t-test
diff_t, diff_p = stats.ttest_rel(df_wide["memory"], df_wide["control"])
if diff_t > 0:
    p_one_sided = diff_p / 2
else:
    p_one_sided = 1 - (diff_p / 2)
print(f"One-sided p-value (memory > control): {p_one_sided:.4f}")

print(f"Paired t-test memory vs control: t = {diff_t:.3f}, p = {diff_p:.4f}, df = {df}")


Mean correlation for memory condition: 0.0391
Mean correlation for control condition: 0.0367

condition   control    memory
subject                      
33        -0.007425  0.014765
34         0.211358  0.066166
35         0.135015 -0.049866
36        -0.036166 -0.069384
38         0.056142  0.023166
Degrees of freedom: 55
One-sided p-value (memory > control): 0.4026
Paired t-test memory vs control: t = 0.248, p = 0.8052, df = 55


In [11]:
%reset -f

### Hold Out 50% with stratifed memory and control

In [17]:
import random
from math import atanh, sqrt
from scipy.stats import norm, ttest_rel, wilcoxon

import matplotlib.pyplot as plt

all_rows = []  # collect all trial-level results here
# best_shift_by_sub = {}  # best shift per subject
rows_trials = []
rows_subject = []  
bad_subjects_mem = []
bad_subjects_ctrl = []

for subj in SUBJECTS:
    print(f"Processing subject {subj:02d}...")
    trials = load_all_trials(subj)                         # list of (eeg, pupil)
    trials_memory, trials_control = split_trials_by_condition(trials)

    random.shuffle(trials_memory)
    random.shuffle(trials_control)

    mid_memory = int(len(trials_memory) // 3)
    mid_control = int(len(trials_control) // 3)

    if mid_memory < 1 or mid_control < 1:
        print(f"Subject {subj:02d} skipped - not enough trials.")
        continue

    train_memory = trials_memory[:mid_memory]
    test_memory  = trials_memory[mid_memory:]
    train_control = trials_control[:mid_control]
    test_control = trials_control[mid_control:]

    #test_memory  = trials_memory
    #test_control  = trials_control

    train_trials = test_memory + test_control
    random.shuffle(train_trials)

    best_shift = best_shift_by_sub[subj] if subj in best_shift_by_sub else print("MISTAKE")
    # ----- fit weights ONCE using all train trials at best_shift ---------
    win = slice(WIN_OFFSET1, -WIN_OFFSET2)

    best_shift = int(best_shift)  # convert to int if it was float
    X_train, Y_train = concat_trials(train_trials, shift=best_shift, win=win)
    X_mem_test, Y_mem_test = concat_trials(test_memory, shift=best_shift, win=win)
    X_ctrl_test, Y_ctrl_test = concat_trials(test_control, shift=best_shift, win=win)

    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(X_train, Y_train)

    # -------------------------------------------------------------
    # 2) whole‑trial correlation for MEMORY
    # -------------------------------------------------------------
    cvX_m, cvY_m = cca.transform(X_mem_test, Y_mem_test)
    r_mem, p_mem = pearsonr(cvX_m[:, 0], cvY_m[:, 0])

    rows_subject.append({ 
        "subject": subj, 
        "condition": "memory", 
        "lag_ms": best_shift * 10, 
        "r": float(r_mem),
        "p-value": float(p_mem)
    })

    # -------------------------------------------------------------
    # 3) whole‑trial correlation for CONTROL
    # -------------------------------------------------------------
    cvX_c, cvY_c = cca.transform(X_ctrl_test, Y_ctrl_test)
    r_ctrl, p_ctrl = pearsonr(cvX_c[:, 0], cvY_c[:, 0])

    rows_subject.append({ 
        "subject": subj, 
        "condition": "control", 
        "lag_ms": best_shift * 10, 
        "r": float(r_ctrl),
        "p-value": float(p_ctrl)
    })

    n_mem  = cvX_m.shape[0]  # number of paired points used in r_mem
    n_ctrl = cvX_c.shape[0]  # number of paired points used in r_ctrl

    z_mem  = atanh(r_mem)
    z_ctrl = atanh(r_ctrl)

    # z-test for H1: r_mem > r_ctrl
    se = sqrt(1/(n_mem - 3) + 1/(n_ctrl - 3))
    z_stat = (z_mem - z_ctrl) / se
    p_one_sided = 1 - norm.cdf(z_stat)

    rows_subject.append({
        "subject": subj,
        "condition": "per-subject test",
        "lag_ms": best_shift * 10,
        "r_mem": float(r_mem),
        "r_ctrl": float(r_ctrl),
        "n_mem": int(n_mem),
        "n_ctrl": int(n_ctrl),
        "z_stat": float(z_stat),
        "p_one_sided": float(p_one_sided),
    })

    if r_ctrl < 0.05:
        bad_subjects_ctrl.append(subj)
    if r_mem < 0.05:
        bad_subjects_mem.append(subj)
    if r_mem < 0.05 and r_ctrl < 0.05:
        print(f"Subject {subj:02d} has both r_mem and r_ctrl < 0.05")

    save_cca_weights(cca, subject_tag=f"sub-{subj:03d}", lag_ms=best_shift*10, condition = "mem", eeg_ch_names=FRONTAL_MIDLINE)

##############################################################################
# ---- save -----------------------------------------------------------------
##############################################################################

##############################################################################
# 4)  after the loop - save per‑subject correlations
##############################################################################
pd.DataFrame(rows_subject).to_csv("subject_level_cca_fixedlagTHESISAllhold3.csv", index=False)
print("Finished - saved subject-level correlations as subject_level_cca_fixedlagTHESISAllhold3.csv.")


Processing subject 33...
saved weights for sub-033 (lag -50 ms)
Processing subject 34...
saved weights for sub-034 (lag +320 ms)
Processing subject 35...
saved weights for sub-035 (lag -80 ms)
Processing subject 36...
saved weights for sub-036 (lag -590 ms)
Processing subject 38...
saved weights for sub-038 (lag -10 ms)
Processing subject 39...
saved weights for sub-039 (lag +20 ms)
Processing subject 40...
saved weights for sub-040 (lag +1000 ms)
Processing subject 41...
saved weights for sub-041 (lag -100 ms)
Processing subject 42...
saved weights for sub-042 (lag -800 ms)
Processing subject 43...
saved weights for sub-043 (lag +700 ms)
Processing subject 44...
saved weights for sub-044 (lag -100 ms)
Processing subject 45...
saved weights for sub-045 (lag +420 ms)
Processing subject 46...
saved weights for sub-046 (lag -680 ms)
Processing subject 47...
saved weights for sub-047 (lag +1000 ms)
Processing subject 48...
saved weights for sub-048 (lag -630 ms)
Processing subject 49...
sa

In [18]:
df = pd.DataFrame(rows_subject)

# keep one row per subject per condition
wide = (df.query("condition in ['memory','control']")
          .pivot(index="subject", columns="condition", values="r"))

# subjects with non-significant per-subject test (p_one_sided > 0.05)
bad_subjects = df.loc[
    (df["condition"] == "per-subject test") & (df["p_one_sided"] > 0.05),
    "subject"
].unique()

print("There are", len (SUBJECTS) - len(bad_subjects), "subjects with significant per-subject test (p_one_sided < 0.05)")

# keep everyone else and make the wide table
wide2 = (df[~df["subject"].isin(bad_subjects)]
          .query("condition in ['memory','control']")
          .pivot(index="subject", columns="condition", values="r"))

# Fisher z
z_mem  = np.arctanh(wide["memory"])
z_ctrl = np.arctanh(wide["control"])
diff   = z_mem - z_ctrl

# one-sided paired t-test: H1 mean(diff) > 0
t2, p_two = ttest_rel(z_mem, z_ctrl, nan_policy='omit')
# convert to one-sided
p_one = p_two / 2 if diff.mean() > 0 else 1 - p_two/2

# (optional) distribution-free paired test on r (or z):
w_stat, p_wilcox_two = wilcoxon(z_mem, z_ctrl, alternative="greater", zero_method="wilcox", correction=False)

print({
    "mean_diff_z": float(np.nanmean(diff)),
    "t_stat": float(t2),
    "p_one_sided_t": float(p_one),
    "wilcoxon_W": float(w_stat),
    "p_one_sided_wilcoxon": float(p_wilcox_two),
    "n_subjects": int(diff.dropna().shape[0]),
})


There are 28 subjects with significant per-subject test (p_one_sided < 0.05)
{'mean_diff_z': 0.013478438708315183, 't_stat': 1.9085864838226698, 'p_one_sided_t': 0.030769752475280233, 'wilcoxon_W': 999.0, 'p_one_sided_wilcoxon': 0.050546889342088945, 'n_subjects': 56}


In [40]:
df_subj = pd.read_csv("subject_level_cca_fixedlagTHESISAllhold3.csv")

# Calculate and print mean correlations for each condition
mean_memory = df_subj[df_subj['condition'] == 'memory']['r'].mean()
mean_control = df_subj[df_subj['condition'] == 'control']['r'].mean()
print(f"Mean correlation for memory condition: {mean_memory:.4f}")
print(f"Mean correlation for control condition: {mean_control:.4f}")
print()

# Pivot so we have columns for memory and control per subject
df_wide = df_subj.pivot(index="subject", columns="condition", values="r")
print(df_wide.head(20))

# --- 95% CI for the paired mean difference (mem − ctl) ---
paired = df_wide.dropna(subset=["memory", "control"])
diff   = paired["memory"] - paired["control"]

n      = diff.size
dfree  = n - 1
mean_d = diff.mean()
se_d   = diff.std(ddof=1) / np.sqrt(n)

alpha  = 0.05
tcrit  = stats.t.ppf(1 - alpha/2, dfree)   # two-sided 95%
ci_low, ci_high = mean_d - tcrit*se_d, mean_d + tcrit*se_d

# (optional) re-run t-test on the same paired subset
t2, p2 = stats.ttest_rel(paired["memory"], paired["control"])
p_one  = p2/2 if t2 > 0 else 1 - p2/2

print(f"[MEM-train only] Paired t-test: t={t2:.3f}, p_two={p2:.4f}, "
      f"p_one(mem>ctl)={p_one:.4f}, df={dfree}, "
      f"meanΔ={mean_d:.3f}, 95% CI=[{ci_low:.3f}, {ci_high:.3f}]")


Mean correlation for memory condition: 0.1051
Mean correlation for control condition: 0.0967

condition   control    memory  per-subject test
subject                                        
33         0.035894  0.063211               NaN
34         0.067763  0.074065               NaN
35         0.157077  0.151807               NaN
36         0.085084  0.076366               NaN
38         0.179091  0.103337               NaN
39         0.083370  0.086907               NaN
40         0.173166  0.108150               NaN
41         0.087716  0.049635               NaN
42         0.000398  0.091751               NaN
43         0.108883  0.138662               NaN
44         0.157053  0.058602               NaN
45         0.064621  0.103250               NaN
46         0.092851  0.197667               NaN
47         0.138267  0.135649               NaN
48         0.073923  0.057150               NaN
49         0.119602  0.102002               NaN
50         0.180887  0.154689             

### Training on 50% memory

In [6]:
import random

import matplotlib.pyplot as plt

all_rows = []  # collect all trial-level results here
# best_shift_by_sub = {}  # best shift per subject
rows_trials = []
rows_subject = []  

for subj in SUBJECTS:
    print(f"Processing subject {subj:02d}...")
    trials = load_all_trials(subj)                         # list of (eeg, pupil)
    trials_memory, trials_control = split_trials_by_condition(trials)

    random.shuffle(trials_memory)
    random.shuffle(trials_control)

    mid_memory = len(trials_memory) // 2
    mid_control = len(trials_control) // 2

    if mid_memory < 1 or mid_control < 1:
        print(f"Subject {subj:02d} skipped - not enough trials.")
        continue

    train_memory = trials_memory[:mid_memory]
    test_memory  = trials_memory[mid_memory:]
    #train_control = trials_control[:mid_control]
    #test_control  = trials_control[mid_control:]

    best_shift = best_shift_by_sub[subj] if subj in best_shift_by_sub else print("MISTAKE")
    # ----- fit weights ONCE using all train trials at best_shift ---------
    win = slice(WIN_OFFSET1, -WIN_OFFSET2)

    best_shift = int(best_shift)  # convert to int if it was float
    X_train, Y_train = concat_trials(train_memory, shift=best_shift, win=win)
    X_mem_test, Y_mem_test = concat_trials(test_memory, shift=best_shift, win=win)
    X_ctrl_test, Y_ctrl_test = concat_trials(trials_control, shift=best_shift, win=win)

    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(X_train, Y_train)

    # -------------------------------------------------------------
    # 2) whole‑trial correlation for MEMORY
    # -------------------------------------------------------------
    cvX_m, cvY_m = cca.transform(X_mem_test, Y_mem_test)
    r_mem, p_mem = pearsonr(cvX_m[:, 0], cvY_m[:, 0])

    rows_subject.append({
        "subject":   subj,
        "condition": "memory",
        "lag_ms":    best_shift * 10,
        "r":         float(r_mem),
        "p-value": float(p_mem),
    })

    # -------------------------------------------------------------
    # 3) whole‑trial correlation for CONTROL
    # -------------------------------------------------------------
    cvX_c, cvY_c = cca.transform(X_ctrl_test, Y_ctrl_test)
    r_ctrl, p_ctrl = pearsonr(cvX_c[:, 0], cvY_c[:, 0])

    rows_subject.append({
        "subject":   subj,
        "condition": "control",
        "lag_ms":    best_shift * 10,
        "r":         float(r_ctrl),
        "p-value": float(p_ctrl), 
    })

    save_cca_weights(cca, subject_tag=f"sub-{subj:03d}", lag_ms=best_shift*10, condition = "mem", eeg_ch_names=FRONTAL_MIDLINE, out_dir=Path("weights_Memhold2"))

##############################################################################
# ---- save -----------------------------------------------------------------
##############################################################################

##############################################################################
# 4)  after the loop - save per‑subject correlations
##############################################################################
pd.DataFrame(rows_subject).to_csv("subject_level_cca_fixedlagTHESISMemhold2.csv", index=False)
print("Finished - saved subject-level correlations as subject_level_cca_fixedlagTHESISMemhold2.csv.")


Processing subject 33...
saved weights for sub-033 (lag -50 ms)
Processing subject 34...
saved weights for sub-034 (lag +320 ms)
Processing subject 35...
saved weights for sub-035 (lag -80 ms)
Processing subject 36...
saved weights for sub-036 (lag -590 ms)
Processing subject 38...
saved weights for sub-038 (lag -10 ms)
Processing subject 39...
saved weights for sub-039 (lag +20 ms)
Processing subject 40...
saved weights for sub-040 (lag +1000 ms)
Processing subject 41...
saved weights for sub-041 (lag -100 ms)
Processing subject 42...
saved weights for sub-042 (lag -800 ms)
Processing subject 43...
saved weights for sub-043 (lag +700 ms)
Processing subject 44...
saved weights for sub-044 (lag -100 ms)
Processing subject 45...
saved weights for sub-045 (lag +420 ms)
Processing subject 46...
saved weights for sub-046 (lag -680 ms)
Processing subject 47...
saved weights for sub-047 (lag +1000 ms)
Processing subject 48...
saved weights for sub-048 (lag -630 ms)
Processing subject 49...
sa

In [53]:
df_subj = pd.read_csv("subject_level_cca_fixedlagTHESISMemhold2.csv")

# Calculate and print mean correlations for each condition
mean_memory = df_subj[df_subj['condition'] == 'memory']['r'].mean()
mean_control = df_subj[df_subj['condition'] == 'control']['r'].mean()
print(f"Mean correlation for memory condition: {mean_memory:.4f}")
print(f"Mean correlation for control condition: {mean_control:.4f}")
print()

# Pivot so we have columns for memory and control per subject
df_wide = df_subj.pivot(index="subject", columns="condition", values="r")
print(df_wide.head())

# Get degrees of freedom
df = len(df_wide) - 1
print(f"Degrees of freedom: {df}")

# Compute paired t-test
diff_t, diff_p = stats.ttest_rel(df_wide["memory"], df_wide["control"])
if diff_t > 0:
    p_one_sided = diff_p / 2
else:
    p_one_sided = 1 - (diff_p / 2)
print(f"One-sided p-value (memory > control): {p_one_sided:.4f}")

print(f"Paired t-test memory vs control: t = {diff_t:.3f}, p = {diff_p:.4f}, df = {df}")


Mean correlation for memory condition: 0.0471
Mean correlation for control condition: 0.0293

condition   control    memory
subject                      
33         0.016894  0.025980
34        -0.102026  0.080174
35        -0.070105  0.030663
36        -0.045677 -0.048873
38         0.121887  0.057935
Degrees of freedom: 55
One-sided p-value (memory > control): 0.0109
Paired t-test memory vs control: t = 2.361, p = 0.0218, df = 55


### Training on 50% mem and ctrl, testing on 100%

In [6]:
import random
from math import atanh, sqrt
from scipy.stats import norm, ttest_rel, wilcoxon

import matplotlib.pyplot as plt

all_rows = []  # collect all trial-level results here
# best_shift_by_sub = {}  # best shift per subject
rows_trials = []
rows_subject = []  
bad_subjects_mem = []
bad_subjects_ctrl = []

for subj in SUBJECTS:
    if subj == 57:
        continue
    print(f"Processing subject {subj:02d}...")
    trials = load_all_trials(subj)                         # list of (eeg, pupil)
    trials_memory, trials_control = split_trials_by_condition(trials)

    random.shuffle(trials_memory)
    random.shuffle(trials_control)

    mid_memory = int(len(trials_memory) // 2)
    mid_control = int(len(trials_control) // 2)

    if mid_memory < 1 or mid_control < 1:
        print(f"Subject {subj:02d} skipped - not enough trials.")
        continue

    train_memory = trials_memory[:mid_memory]
    test_memory  = trials_memory

    train_control = trials_control[:mid_memory]
    test_control = trials_control

    train_trials = train_memory + train_control
    random.shuffle(train_trials)

    best_shift = best_shift_by_sub[subj] if subj in best_shift_by_sub else print("MISTAKE")
    # ----- fit weights ONCE using all train trials at best_shift ---------
    win = slice(WIN_OFFSET1, -WIN_OFFSET2)

    best_shift = int(best_shift)  # convert to int if it was float
    X_train, Y_train = concat_trials(train_trials, shift=best_shift, win=win)
    X_mem_test, Y_mem_test = concat_trials(test_memory, shift=best_shift, win=win)
    X_ctrl_test, Y_ctrl_test = concat_trials(test_control, shift=best_shift, win=win)



    cca = CCA(n_components=1, max_iter=1000, scale=False, tol=1e-6)
    cca.fit(X_train, Y_train)

    # -------------------------------------------------------------
    # 2) whole‑trial correlation for MEMORY
    # -------------------------------------------------------------
    cvX_m, cvY_m = cca.transform(X_mem_test, Y_mem_test)
    r_mem, p_mem = pearsonr(cvX_m[:, 0], cvY_m[:, 0])

    rows_subject.append({
        "subject":   subj,
        "condition": "memory",
        "lag_ms":    best_shift * 10,
        "r":         float(r_mem),
        "p-value": float(p_mem),
    })

    # -------------------------------------------------------------
    # 3) whole‑trial correlation for CONTROL
    # -------------------------------------------------------------
    cvX_c, cvY_c = cca.transform(X_ctrl_test, Y_ctrl_test)
    r_ctrl, p_ctrl = pearsonr(cvX_c[:, 0], cvY_c[:, 0])

    rows_subject.append({
        "subject":   subj,
        "condition": "control",
        "lag_ms":    best_shift * 10,
        "r":         float(r_ctrl),
        "p-value": float(p_ctrl), 
    })

    save_cca_weights(cca, subject_tag=f"sub-{subj:03d}", lag_ms=best_shift*10, condition = "mem", eeg_ch_names=FRONTAL_MIDLINE, out_dir=Path("weights_Memhold2"))


    n_mem  = cvX_m.shape[0]  # number of paired points used in r_mem
    n_ctrl = cvX_c.shape[0]  # number of paired points used in r_ctrl

    z_mem  = atanh(r_mem)
    z_ctrl = atanh(r_ctrl)

    # z-test for H1: r_mem > r_ctrl
    se = sqrt(1/(n_mem - 3) + 1/(n_ctrl - 3))
    z_stat = (z_mem - z_ctrl) / se
    p_one_sided = 1 - norm.cdf(z_stat)

    rows_subject.append({
        "subject": subj,
        "condition": "per-subject test",
        "lag_ms": best_shift * 10,
        "r_mem": float(r_mem),
        "r_ctrl": float(r_ctrl),
        "n_mem": int(n_mem),
        "n_ctrl": int(n_ctrl),
        "z_stat": float(z_stat),
        "p_one_sided": float(p_one_sided),
    })

    if r_ctrl < 0.05:
        bad_subjects_ctrl.append(subj)
    if r_mem < 0.05:
        bad_subjects_mem.append(subj)
    if r_mem < 0.05 and r_ctrl < 0.05:
        print(f"Subject {subj:02d} has both r_mem and r_ctrl < 0.05")

    save_cca_weights(cca, subject_tag=f"sub-{subj:03d}", lag_ms=best_shift*10, condition = "mem", eeg_ch_names=FRONTAL_MIDLINE, out_dir=Path("weights50_100"))

##############################################################################
# ---- save -----------------------------------------------------------------
##############################################################################

##############################################################################
# 4)  after the loop - save per‑subject correlations
##############################################################################
pd.DataFrame(rows_subject).to_csv("subject_level_cca_fixedlag50_100.csv", index=False)
print("Finished - saved subject-level correlations as subject_level_cca_fixedlag50_100.csv.")


Processing subject 33...
saved weights for sub-033 (lag -50 ms)
Subject 33 has both r_mem and r_ctrl < 0.05
saved weights for sub-033 (lag -50 ms)
Processing subject 34...
saved weights for sub-034 (lag +320 ms)
saved weights for sub-034 (lag +320 ms)
Processing subject 35...
saved weights for sub-035 (lag -80 ms)
saved weights for sub-035 (lag -80 ms)
Processing subject 36...
saved weights for sub-036 (lag -590 ms)
Subject 36 has both r_mem and r_ctrl < 0.05
saved weights for sub-036 (lag -590 ms)
Processing subject 38...
saved weights for sub-038 (lag -10 ms)
saved weights for sub-038 (lag -10 ms)
Processing subject 39...
saved weights for sub-039 (lag +20 ms)
saved weights for sub-039 (lag +20 ms)
Processing subject 40...
saved weights for sub-040 (lag +1000 ms)
saved weights for sub-040 (lag +1000 ms)
Processing subject 41...
saved weights for sub-041 (lag -100 ms)
saved weights for sub-041 (lag -100 ms)
Processing subject 42...
saved weights for sub-042 (lag -800 ms)
saved weights

In [7]:
df = pd.DataFrame(rows_subject)

# keep one row per subject per condition
wide = (df.query("condition in ['memory','control']")
          .pivot(index="subject", columns="condition", values="r"))

# subjects with non-significant per-subject test (p_one_sided > 0.05)
bad_subjects = df.loc[
    (df["condition"] == "per-subject test") & (df["p_one_sided"] > 0.05),
    "subject"
].unique()

print("There are", len (SUBJECTS) - len(bad_subjects), "subjects with significant per-subject test (p_one_sided < 0.05)")

# keep everyone else and make the wide table
wide2 = (df[~df["subject"].isin(bad_subjects)]
          .query("condition in ['memory','control']")
          .pivot(index="subject", columns="condition", values="r"))

# Fisher z
z_mem  = np.arctanh(wide["memory"])
z_ctrl = np.arctanh(wide["control"])
diff   = z_mem - z_ctrl

# one-sided paired t-test: H1 mean(diff) > 0
t2, p_two = ttest_rel(z_mem, z_ctrl, nan_policy='omit')
# convert to one-sided
p_one = p_two / 2 if diff.mean() > 0 else 1 - p_two/2

# (optional) distribution-free paired test on r (or z):
w_stat, p_wilcox_two = wilcoxon(z_mem, z_ctrl, alternative="greater", zero_method="wilcox", correction=False)

print({
    "mean_diff_z": float(np.nanmean(diff)),
    "t_stat": float(t2),
    "p_one_sided_t": float(p_one),
    "wilcoxon_W": float(w_stat),
    "p_one_sided_wilcoxon": float(p_wilcox_two),
    "n_subjects": int(diff.dropna().shape[0]),
})


There are 19 subjects with significant per-subject test (p_one_sided < 0.05)
{'mean_diff_z': -0.018627798826517855, 't_stat': -2.5943906340358938, 'p_one_sided_t': 0.99391312024824, 'wilcoxon_W': 471.0, 'p_one_sided_wilcoxon': 0.9938806478590789, 'n_subjects': 55}
